# LinQ · Две модели параллельно: две независимые GPU

**Бюджет — до 4–5 часов на КАЖДОЕ направление.** Два задания не делят один тренировочный бюджет.
Нужен соответствующий `linq-person-training-input.zip` / `linq-clothing-training-input.zip`, добавленный как Kaggle Dataset → Add Input.
Для `both` добавьте оба архива и включите **GPU T4 ×2**. При одной GPU используйте один из отдельных ноутбуков.

250 минут — потолок оптимизации каждой модели. Валидация и экспорт используют резерв; внешний предел 285 минут.
Ранняя остановка допустима: расходовать все часы после устойчивого плато не требуется.
Чекпойнты и логи сохраняются в `/kaggle/working/targeted-results`; результаты нужно сохранить при завершении.
Копии внутри одной VM не защищают от её уничтожения. Запись в отдельный Dataset/API отключена по выбору владельца.

Обучение использует только train/val. Test не копируется в рабочий тренировочный набор и не участвует в подборе весов/порогов.
Разметка новых фабричных примеров выполнена ассистентом и не заменяет независимую экспертную оценку.
Ни один выходной файл не устанавливается в приложение автоматически. Совместимость ONNX-графа проверяется; компиляция LinQ требует отдельной проверки.


In [ ]:
import os,sys,time,json,zipfile,subprocess,shutil,hashlib
from pathlib import Path
NOTEBOOK_STARTED=time.monotonic()
TASK='both'
TRAIN_MINUTES=250
# Optional: optimizer checkpoints from your own previous Outputs. Set remaining
# TRAIN_MINUTES explicitly on resume, and attach sibling best.pt as well.
RESUME_CHECKPOINTS={'person':None,'clothing':None}
os.environ['PYTHONUNBUFFERED']='1'
os.environ['OMP_NUM_THREADS']='2';os.environ['MKL_NUM_THREADS']='2'
print('SETUP_STARTED',TASK,flush=True)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.88','onnx==1.19.1','onnxruntime>=1.20,<2','PyYAML>=6.0'],check=True)
import torch
assert torch.cuda.is_available(),'Kaggle Settings: enable GPU before Save & Run All'
print('GPU_READY',torch.cuda.device_count(),[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],flush=True)
if TASK=='both':assert torch.cuda.device_count()>=2,'For both tasks use T4 x2; otherwise run separate notebooks, each with its own 4-5h budget.'
PROGRAM=Path('/kaggle/working/targeted-program');PROGRAM.mkdir(exist_ok=True)
RESULTS=Path('/kaggle/working/targeted-results');RESULTS.mkdir(exist_ok=True)


In [ ]:
PROGRAM_FILES={'train_targeted_person.py': '"""Bounded YOLO12m person adaptation. A candidate, never an auto-deployment."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport time\n\nfrom training_checkpoint import atomic_copy\nfrom training_observability import Progress,atomic_json\n\n\ndef main():\n    p=argparse.ArgumentParser()\n    p.add_argument(\'--data\',type=Path,required=True);p.add_argument(\'--weights\',type=Path,required=True)\n    p.add_argument(\'--output\',type=Path,required=True);p.add_argument(\'--minutes\',type=float,default=250)\n    p.add_argument(\'--epochs\',type=int,default=100);p.add_argument(\'--device\',default=\'0\')\n    p.add_argument(\'--batch\',type=int,default=16);p.add_argument(\'--resume\',type=Path)\n    p.add_argument(\'--no-amp\',action=\'store_true\')\n    a=p.parse_args();a.data=a.data.resolve();a.weights=a.weights.resolve();a.output=a.output.resolve()\n    if a.resume:a.resume=a.resume.resolve()\n    a.output.mkdir(parents=True,exist_ok=True)\n    progress=Progress(a.output/\'progress.json\')\n    try:\n        import torch,cv2,yaml\n        from ultralytics import YOLO,settings\n        torch.set_num_threads(2);cv2.setNumThreads(1)\n        torch.multiprocessing.set_start_method(\'spawn\',force=True)\n        dict.update(settings,{k:False for k in [\'wandb\',\'mlflow\',\'comet\',\'clearml\',\'neptune\',\'raytune\',\'hub\'] if k in settings})\n        # Retain COCO\'s initialized person output instead of reinitializing a\n        # single-class head. Only class zero is a valid deployment output here.\n        model=YOLO(str(a.resume or a.weights));names=dict(model.names)\n        if a.resume:\n            if model.ckpt.get(\'epoch\',-1)<0 or not model.ckpt.get(\'optimizer\'):\n                raise ValueError(\'Use resume-last.pt with optimizer, not exported best.pt\')\n        if len(names)!=80 or names.get(0)!=\'person\':raise ValueError(\'Expected COCO YOLO12m person=0, not phone/head weights\')\n        data=a.output/\'person-data.yaml\'\n        data.write_text(yaml.safe_dump({\'path\':str(a.data.resolve()),\'train\':\'images/train\',\'val\':\'images/val\',\'names\':names}),encoding=\'utf-8\')\n        if a.resume:\n            prior=a.resume.parent/\'best.pt\';target=a.output/\'person/weights/best.pt\'\n            if not prior.exists():raise ValueError(\'Attach the prior best.pt beside the optimizer checkpoint\')\n            if prior.resolve()!=target.resolve():atomic_copy(prior,target)\n        clock={};counts={\'batch\':0,\'epoch\':0}\n        def begin(trainer):clock[\'end\']=time.monotonic()+a.minutes*60\n        def epoch(trainer):\n            counts.update(batch=0,epoch=int(trainer.epoch)+1)\n            progress.update(\'train\',epoch=counts[\'epoch\'],batch=0,total_batches=len(trainer.train_loader),force=True)\n        def batch(trainer):\n            counts[\'batch\']+=1\n            progress.update(\'train\',epoch=counts[\'epoch\'],batch=counts[\'batch\'],total_batches=len(trainer.train_loader))\n            if time.monotonic()>=clock[\'end\'] or (a.output/\'STOP\').exists():trainer.stop=True\n        def save(trainer):\n            if not trainer.last.exists():return\n            target=trainer.last.with_name(\'resume-last.pt\')\n            if target.exists():atomic_copy(target,target.with_name(\'resume-previous.pt\'))\n            atomic_copy(trainer.last,target)\n            state=torch.load(target,map_location=\'cpu\',weights_only=False)  # Our just-created checkpoint.\n            if not state.get(\'optimizer\'):raise ValueError(\'Optimizer missing from checkpoint\')\n            proof={\'epoch_index\':int(state[\'epoch\']),\'optimizer_present\':True,\'sha256\':hashlib.sha256(target.read_bytes()).hexdigest(),\'bytes\':target.stat().st_size}\n            del state;atomic_json(target.parent/\'checkpoint-state.json\',proof)\n            print(\'CHECKPOINT_SAVED\',json.dumps(proof),flush=True)\n        for event,fn in [(\'on_train_start\',begin),(\'on_train_epoch_start\',epoch),(\'on_train_batch_end\',batch),(\'on_model_save\',save)]:model.add_callback(event,fn)\n        # Low-rate adaptation with the earliest features frozen limits forgetting\n        # of ordinary/occluded COCO people. Tiny-object variants are train-only.\n        progress.update(\'initializing\',force=True)\n        model.train(data=str(data),epochs=a.epochs,imgsz=640,batch=a.batch,device=a.device,\n            workers=2,cache=False,amp=not a.no_amp,optimizer=\'AdamW\',lr0=0.00008,lrf=.05,\n            warmup_epochs=2,warmup_bias_lr=0.,freeze=10,weight_decay=.0005,\n            mosaic=.35,close_mosaic=10,mixup=0.,copy_paste=0.,scale=.55,\n            translate=.08,fliplr=.5,flipud=0.,degrees=0.,hsv_h=.01,hsv_s=.3,hsv_v=.3,\n            patience=15,seed=42,project=str(a.output),name=\'person\',exist_ok=True,\n            save_period=-1,resume=bool(a.resume))\n        best=a.output/\'person/weights/best.pt\'\n        if not best.exists():raise RuntimeError(\'No validated best checkpoint; resume weights remain preserved\')\n        candidate=YOLO(str(best))\n        progress.update(\'final_validation\',force=True)\n        metrics=candidate.val(data=str(data),split=\'val\',device=a.device,batch=a.batch,workers=2,imgsz=640,verbose=False)\n        per_class={str(k):list(map(float,metrics.box.class_result(i))) for i,k in enumerate(metrics.box.ap_class_index)}\n        progress.update(\'export\',force=True)\n        exported=Path(candidate.export(format=\'onnx\',imgsz=640,batch=1,dynamic=False,nms=False,opset=12,simplify=False,device=\'cpu\'))\n        target=a.output/\'person-yolo12m-candidate.onnx\';atomic_copy(exported,target)\n        import onnx\n        onnx.checker.check_model(str(target))\n        report={\'task\':\'person\',\'architecture\':\'YOLO12m COCO-80 head; allowed output person=0\',\n            \'validation\':{k:float(v) for k,v in metrics.results_dict.items()},\'per_class_P_R_AP50_AP5095\':per_class,\n            \'training_budget_minutes\':a.minutes,\'deployment_allowed_classes\':[0],\n            \'onnx_sha256\':hashlib.sha256(target.read_bytes()).hexdigest(),\n            \'status\':\'candidate_not_deployed\',\'test_evaluated\':False,\'linq_compiled\':False,\'onnx_checker_passed\':True,\n            \'required_gate\':\'Compare matched people, machine false positives, small/occluded recall, track continuity and latency on fixed reviewed validation. Keep old weights if regression.\',\n            \'warning\':\'Validation and augmented views are not independent factory event accuracy.\'}\n        atomic_json(a.output/\'candidate.json\',report);progress.update(\'complete\',force=True)\n        print(\'CANDIDATE_READY\',json.dumps(report,default=float),flush=True)\n    except BaseException as e:\n        progress.update(\'failed\',error=type(e).__name__,force=True);raise\n    finally:progress.close()\n\n\nif __name__==\'__main__\':main()\n', 'train_signal_clothing.py': '"""Three-way clothing verifier: visible signal garment / ordinary / unobservable.\n\nReflective work uniforms are compliant by the owner\'s stated policy. Upstream\nno_vest labels are NOT automatically ordinary-clothing ground truth.\n"""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport random\nimport time\n\nfrom training_checkpoint import atomic_torch_save,atomic_copy\nfrom training_observability import Progress,atomic_json\n\nCLASSES=[\'hi_vis\',\'ordinary\',\'unobservable\']\n\n\ndef balanced_weights(rows,factory_share=.35):\n    """Balance classes and source groups; reserve a bounded factory share.\n\n    Otherwise a few reflective uniforms are overwhelmed by upstream bright\n    vests. More crops from one source image/video are not independent groups.\n    """\n    from collections import Counter,defaultdict\n    counts=Counter();groups=defaultdict(set);domains=defaultdict(set)\n    for row in rows:\n        domain=\'factory\' if row[\'source_group\'].startswith(\'factory:\') else \'upstream\'\n        key=(row[\'label\'],domain,row[\'source_group\']);counts[key]+=1\n        groups[row[\'label\'],domain].add(row[\'source_group\']);domains[row[\'label\']].add(domain)\n    result=[]\n    for row in rows:\n        label=row[\'label\'];domain=\'factory\' if row[\'source_group\'].startswith(\'factory:\') else \'upstream\'\n        mass=(factory_share if domain==\'factory\' else 1-factory_share) if len(domains[label])==2 else 1.\n        result.append(mass/len(CLASSES)/len(groups[label,domain])/counts[label,domain,row[\'source_group\']])\n    return result\n\n\ndef classification_report(scores,truth,thresholds):\n    import numpy as np\n    pred=scores.argmax(1);accepted=scores.max(1)>=np.asarray(thresholds)[pred]\n    # An unobservable prediction is an explicit abstention, never a violation.\n    result={};matrix=np.zeros((3,4),dtype=int)\n    for y,p,ok in zip(truth,pred,accepted):matrix[int(y),int(p) if ok else 3]+=1\n    for k,name in enumerate(CLASSES):\n        tp=int(((truth==k)&(pred==k)&accepted).sum());fp=int(((truth!=k)&(pred==k)&accepted).sum())\n        fn=int(((truth==k)&((pred!=k)|~accepted)).sum());pr=tp/max(1,tp+fp);re=tp/max(1,tp+fn)\n        result[name]={\'tp\':tp,\'fp\':fp,\'fn\':fn,\'precision\':pr,\'recall\':re,\'f1\':2*pr*re/max(1e-12,pr+re)}\n    return {\'classes\':result,\'confusion_truth_by_prediction_or_abstain\':matrix.tolist(),\n        \'coverage\':float(accepted.mean()),\'actionable_coverage\':float((accepted&(pred!=2)).mean()),\n        \'macro_f1\':float(np.mean([r[\'f1\'] for r in result.values()]))}\n\n\ndef calibrate(scores,truth):\n    import numpy as np\n    pred=scores.argmax(1);thresholds=[];passed=[]\n    for k in range(3):\n        choices=[]\n        for t in np.linspace(.8,.995,40):\n            mask=(pred==k)&(scores[:,k]>=t);n=int(mask.sum());precision=float((truth[mask]==k).mean()) if n else 0.\n            if n>=10 and precision>=(.98 if k==1 else .95):choices.append((n,float(t)))\n        thresholds.append(max(choices)[1] if choices else 1.01);passed.append(bool(choices))\n    return thresholds,passed\n\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--data\',type=Path,required=True)\n    p.add_argument(\'--weights\',type=Path,required=True);p.add_argument(\'--output\',type=Path,required=True)\n    p.add_argument(\'--minutes\',type=float,default=250);p.add_argument(\'--epochs\',type=int,default=200)\n    p.add_argument(\'--device\',default=\'0\');p.add_argument(\'--batch\',type=int,default=64);p.add_argument(\'--resume\',type=Path)\n    a=p.parse_args();a.data=a.data.resolve();a.weights=a.weights.resolve();a.output=a.output.resolve()\n    if a.resume:a.resume=a.resume.resolve()\n    a.output.mkdir(parents=True,exist_ok=True);progress=Progress(a.output/\'progress.json\')\n    try:\n        import numpy as np\n        import torch\n        from torchvision import models,transforms\n        from torch.utils.data import DataLoader,WeightedRandomSampler,Dataset\n        from PIL import Image\n        torch.set_num_threads(2);torch.manual_seed(42);random.seed(42);np.random.seed(42)\n        torch.multiprocessing.set_start_method(\'spawn\',force=True)\n        device=torch.device(\'cuda:\'+a.device if a.device.isdigit() else a.device)\n        # Loaders are kept in-process: no fork of initialized CUDA, heartbeat\n        # threads or mutable random state. The 224px crops are staged locally.\n        normalize=transforms.Normalize([.485,.456,.406],[.229,.224,.225])\n        base=transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),normalize])\n        augment=transforms.Compose([transforms.Resize((224,224)),transforms.RandomHorizontalFlip(),\n            transforms.RandomAffine(5,translate=(.04,.04),scale=(.9,1.05)),\n            transforms.ColorJitter(.12,.12,.1,.015),transforms.ToTensor(),normalize])\n        class Rows(Dataset):\n            def __init__(self,split,train=False):\n                self.rows=[r for r in map(json.loads,(a.data/\'manifest.jsonl\').read_text().splitlines()) if r[\'split\']==split]\n                self.transform=augment if train else base\n                self.targets=[CLASSES.index(r[\'label\']) for r in self.rows]\n            def __len__(self):return len(self.rows)\n            def __getitem__(self,index):\n                row=self.rows[index]\n                with Image.open(a.data/row[\'file\']) as im:image=im.convert(\'RGB\')\n                return self.transform(image),self.targets[index]\n        train=Rows(\'train\',True);val=Rows(\'val\')\n        for data in [train,val]:\n            counts=np.bincount(data.targets,minlength=3)\n            if counts.min()<5:raise ValueError(\'Each class needs at least five real reviewed samples per training/validation split\')\n        model=models.resnet50(weights=None)\n        checkpoint=torch.load(a.weights,map_location=\'cpu\',weights_only=True)\n        state=checkpoint.get(\'state_dict\',checkpoint)\n        state={k:v for k,v in state.items() if not k.startswith(\'fc.\')}\n        model.fc=torch.nn.Linear(2048,3);model.load_state_dict(state,strict=False)\n        # The old binary fc is deliberately discarded: no_vest is not no_hi_vis.\n        for name,param in model.named_parameters():param.requires_grad=name.startswith((\'layer3.\',\'layer4.\',\'fc.\'))\n        model.to(device)\n        optimizer=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=3e-5,weight_decay=.001)\n        scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=a.epochs,eta_min=2e-6)\n        scaler=torch.amp.GradScaler(\'cuda\',enabled=device.type==\'cuda\')\n        sampling=torch.tensor(balanced_weights(train.rows),dtype=torch.double)\n        start_epoch=0;best=-1.;bad=0\n        if a.resume:\n            saved=torch.load(a.resume,map_location=device,weights_only=True)\n            if saved[\'classes\']!=CLASSES:raise ValueError(\'Resume classes differ\')\n            model.load_state_dict(saved[\'state_dict\']);optimizer.load_state_dict(saved[\'optimizer\'])\n            scaler.load_state_dict(saved[\'scaler\']);scheduler.load_state_dict(saved[\'scheduler\'])\n            start_epoch=int(saved[\'next_epoch\']);best=saved[\'best_score\'];bad=saved.get(\'bad_epochs\',0)\n            torch.set_rng_state(saved[\'torch_rng\'].cpu());random.setstate(saved[\'python_rng\'])\n            if device.type==\'cuda\' and saved.get(\'cuda_rng\'):torch.cuda.set_rng_state_all([x.cpu() for x in saved[\'cuda_rng\']])\n            prior=a.resume.parent/\'best.pt\'\n            if not prior.exists():raise ValueError(\'Attach the prior best.pt beside the optimizer checkpoint\')\n            if prior.resolve()!=(a.output/\'best.pt\').resolve():atomic_copy(prior,a.output/\'best.pt\')\n        loss_fn=torch.nn.CrossEntropyLoss(label_smoothing=.03)\n        val_loader=DataLoader(val,batch_size=a.batch,num_workers=0)\n        def evaluate():\n            model.eval();scores=[];truth=[]\n            with torch.inference_mode():\n                for x,y in val_loader:scores.append(model(x.to(device)).softmax(1).cpu().numpy());truth.extend(y.tolist())\n            return np.concatenate(scores),np.asarray(truth)\n        def save(path,next_epoch,partial=False):\n            payload={\'classes\':CLASSES,\'architecture\':\'resnet50\',\'state_dict\':model.state_dict(),\n                \'optimizer\':optimizer.state_dict(),\'scaler\':scaler.state_dict(),\'scheduler\':scheduler.state_dict(),\n                \'next_epoch\':next_epoch,\'partial_epoch_replayed_on_resume\':partial,\'best_score\':best,\'bad_epochs\':bad,\n                \'torch_rng\':torch.get_rng_state(),\'python_rng\':random.getstate(),\n                \'cuda_rng\':torch.cuda.get_rng_state_all() if device.type==\'cuda\' else []}\n            atomic_torch_save(payload,path,previous=True)\n            print(\'CHECKPOINT_SAVED\',path.name,next_epoch,partial,flush=True)\n        deadline=time.monotonic()+a.minutes*60;began=time.monotonic();last_recovery=began\n        history=[]\n        for epoch in range(start_epoch,a.epochs):\n            generator=torch.Generator().manual_seed(42+epoch)\n            sampler=WeightedRandomSampler(sampling,len(train),replacement=True,generator=generator)\n            loader=DataLoader(train,batch_size=a.batch,sampler=sampler,num_workers=0)\n            model.train();model.conv1.eval();model.bn1.eval();model.layer1.eval();model.layer2.eval()\n            progress.update(\'train\',epoch=epoch+1,batch=0,total_batches=len(loader),force=True)\n            for step,(x,y) in enumerate(loader,1):\n                optimizer.zero_grad(set_to_none=True)\n                with torch.autocast(device_type=device.type,enabled=device.type==\'cuda\'):loss=loss_fn(model(x.to(device)),y.to(device))\n                scaler.scale(loss).backward();scaler.step(optimizer);scaler.update()\n                progress.update(\'train\',epoch=epoch+1,batch=step,total_batches=len(loader),loss=float(loss.detach()))\n                if time.monotonic()-last_recovery>=300:\n                    save(a.output/\'recovery-last.pt\',epoch,partial=True);last_recovery=time.monotonic()\n                if time.monotonic()>=deadline or (a.output/\'STOP\').exists():break\n            progress.update(\'validation\',epoch=epoch+1,force=True)\n            scores,truth=evaluate();metrics=classification_report(scores,truth,[0.,0.,0.])\n            score=metrics[\'macro_f1\'];scheduler.step()\n            if score>best+1e-5:\n                best=score;bad=0\n                atomic_torch_save({\'state_dict\':model.state_dict(),\'classes\':CLASSES,\'epoch\':epoch+1},a.output/\'best.pt\')\n            else:bad+=1\n            history.append({\'epoch\':epoch+1,\'validation\':metrics,\'loss_last_batch\':float(loss.detach())})\n            atomic_json(a.output/\'history.json\',history);save(a.output/\'resume-last.pt\',epoch+1)\n            print(\'EPOCH_RESULT\',json.dumps(history[-1]),flush=True)\n            if time.monotonic()>=deadline or (a.output/\'STOP\').exists() or (time.monotonic()-began>90*60 and bad>=15):break\n        saved=torch.load(a.output/\'best.pt\',map_location=device,weights_only=True);model.load_state_dict(saved[\'state_dict\'])\n        scores,truth=evaluate();thresholds,passed=calibrate(scores,truth)\n        report={\'task\':\'signal_clothing\',\'architecture\':\'ResNet50 / three output logits\',\'classes\':CLASSES,\n            \'validation\':classification_report(scores,truth,thresholds),\'thresholds\':thresholds,\n            \'validation_precision_gate_passed\':passed,\'best_epoch\':saved[\'epoch\'],\'training_budget_minutes\':a.minutes,\n            \'test_evaluated\':False,\'status\':\'candidate_not_deployed\',\'linq_compiled\':False,\n            \'policy\':\'Bright signal vest OR visually identifiable reflective work uniform accepted. Unobservable is abstention.\',\n            \'sampler\':\'Class and source-group balance; factory share capped at 35% within each class when both domains are present.\',\n            \'warning\':\'Assistant-reviewed development labels, not independent expert safety certification. A threshold above 1 disables that class when validation support/precision is insufficient.\'}\n        progress.update(\'export\',force=True);model.eval().cpu()\n        target=a.output/\'signal-clothing-candidate.onnx\'\n        torch.onnx.export(model,torch.zeros(1,3,224,224),str(target),input_names=[\'images\'],output_names=[\'logits\'],opset_version=12,dynamo=False)\n        import onnx\n        graph=onnx.load(target)\n        metadata={\'class_names\':json.dumps(CLASSES),\'class_thresholds\':json.dumps(thresholds),\n            \'rgb_mean\':json.dumps([.485,.456,.406]),\'rgb_std\':json.dumps([.229,.224,.225]),\n            \'crop_policy\':\'person_body_context_1.05\',\'resize_algorithm\':\'pil_bilinear\',\n            \'policy\':\'vest_or_reflective_uniform\',\'candidate_status\':\'not_deployed\'}\n        for key,value in metadata.items():entry=graph.metadata_props.add();entry.key=key;entry.value=value\n        onnx.checker.check_model(graph);onnx.save(graph,target)\n        import onnxruntime as ort\n        options=ort.SessionOptions();options.intra_op_num_threads=2;options.inter_op_num_threads=1\n        options.add_session_config_entry(\'session.intra_op.allow_spinning\',\'0\')\n        session=ort.InferenceSession(str(target),sess_options=options,providers=[\'CPUExecutionProvider\'])\n        sample=np.zeros((1,3,224,224),np.float32)\n        with torch.inference_mode():reference=model(torch.from_numpy(sample)).numpy()\n        actual=session.run(None,{\'images\':sample})[0]\n        error=float(np.abs(reference-actual).max())\n        if error>1e-3:raise ValueError(\'Exported ONNX logits fail FP32 parity\')\n        report[\'onnx_fp32_parity_max_error\']=error\n        report[\'onnx_sha256\']=hashlib.sha256(target.read_bytes()).hexdigest()\n        atomic_json(a.output/\'candidate.json\',report);progress.update(\'complete\',force=True)\n        print(\'CANDIDATE_READY\',json.dumps(report),flush=True)\n    except BaseException as error:progress.update(\'failed\',error=type(error).__name__,force=True);raise\n    finally:progress.close()\n\n\nif __name__==\'__main__\':main()\n', 'training_checkpoint.py': '"""Crash-safe replacement of our own training checkpoints on one filesystem."""\nimport os\nfrom pathlib import Path\nimport shutil\n\n\ndef atomic_copy(source, target):\n    target = Path(target)\n    target.parent.mkdir(parents=True, exist_ok=True)\n    temporary = target.with_name(target.name + \'.writing\')\n    with Path(source).open(\'rb\') as incoming, temporary.open(\'wb\') as outgoing:\n        shutil.copyfileobj(incoming, outgoing)\n        outgoing.flush()\n        os.fsync(outgoing.fileno())\n    os.replace(temporary, target)\n\n\ndef atomic_torch_save(state, target, previous=False):\n    import torch\n    target = Path(target)\n    target.parent.mkdir(parents=True, exist_ok=True)\n    temporary = target.with_name(target.name + \'.writing\')\n    with temporary.open(\'wb\') as stream:\n        torch.save(state, stream)\n        stream.flush()\n        os.fsync(stream.fileno())\n    if previous and target.exists():\n        atomic_copy(target, target.with_name(target.stem + \'-previous.pt\'))\n    os.replace(temporary, target)\n', 'training_observability.py': '"""Small atomic progress records and bounded local staging for prepared FPI data."""\nfrom concurrent.futures import FIRST_COMPLETED,ThreadPoolExecutor,wait\nfrom datetime import datetime,timezone\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport shutil\nimport threading\nimport time\n\n\ndef atomic_json(path,payload):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    temporary=path.with_name(path.name+\'.writing\')\n    temporary.write_text(json.dumps(payload,ensure_ascii=False),encoding=\'utf-8\')\n    # Windows readers can briefly hold a non-delete-shared handle. Linux/Kaggle\n    # replaces immediately; bounded retries keep local monitoring atomic too.\n    for attempt in range(8):\n        try:os.replace(temporary,path);break\n        except PermissionError:\n            if attempt==7:raise\n            time.sleep(min(.01*2**attempt,.1))\n\n\nclass Progress:\n    """Heartbeat means the Python process is alive, not that GPU work advances."""\n    def __init__(self,path,interval=10):\n        self.path=Path(path);self.interval=interval;self.lock=threading.RLock()\n        self.started=time.monotonic();self.work_at=self.started;self.phase_at=self.started;self.written=0\n        self.payload={\'stage\':\'starting\',\'pid\':os.getpid(),\'sequence\':0};self.stop=threading.Event()\n        self.thread=threading.Thread(target=self._heartbeat,name=\'training-heartbeat\',daemon=True)\n        self.update(\'starting\',force=True);self.thread.start()\n    def _write(self):\n        now=time.monotonic();self.payload[\'sequence\']+=1\n        atomic_json(self.path,{**self.payload,\'heartbeat_utc\':datetime.now(timezone.utc).isoformat(),\n            \'elapsed_s\':round(now-self.started,1),\'stage_elapsed_s\':round(now-self.phase_at,1),\n            \'last_work_age_s\':round(now-self.work_at,1)})\n        self.written=now\n    def update(self,stage=None,force=False,**fields):\n        with self.lock:\n            changed=stage is not None and stage!=self.payload[\'stage\']\n            if changed:\n                self.phase_at=time.monotonic()\n                self.payload={key:self.payload[key] for key in [\'stage\',\'pid\',\'sequence\']}\n            if stage is not None:self.payload[\'stage\']=stage\n            self.payload.update(fields);self.work_at=time.monotonic()\n            if changed or force or self.work_at-self.written>=2:self._write()\n            if changed:print(\'TRAINING_STAGE\',self.payload[\'stage\'],json.dumps(fields),flush=True)\n    def _heartbeat(self):\n        while not self.stop.wait(self.interval):\n            with self.lock:self._write()\n    def close(self):\n        self.stop.set();self.thread.join(timeout=2)\n        with self.lock:self._write()\n\n\ndef remap_labels(text):\n    rows=[]\n    for line in text.splitlines():\n        if not line.strip():continue\n        fields=line.split()\n        if len(fields)!=5 or fields[0] not in {\'0\',\'1\'}:raise ValueError(\'Use normalized prepared FPI labels\')\n        values=[float(v) for v in fields[1:]]\n        if any(not math.isfinite(v) or not 0<=v<=1 for v in values) or min(values[2:])<=0:\n            raise ValueError(\'Invalid normalized FPI box\')\n        fields[0]=\'67\' if fields[0]==\'0\' else \'0\';rows.append(\' \'.join(fields))\n    return \'\\n\'.join(rows)+\'\\n\'\n\n\ndef materialize_fpi(source,mapped,progress,workers=8,include_test=False):\n    """Copy bytes locally once; workers never resolve per-file remote paths."""\n    source=Path(source);mapped=Path(mapped)\n    if not source.is_absolute():raise ValueError(\'Resolve the input root once before staging\')\n    mapped.mkdir(parents=True,exist_ok=True);marker=mapped/\'materialization.json\'\n    identity=hashlib.sha256(str(source).encode())\n    for name in [\'FPI_PREPARED_SOURCE.json\',\'manifest.json\']:\n        file=source/name\n        if file.exists():identity.update(file.read_bytes())  # Two metadata reads, not one per image.\n    identity=identity.hexdigest();prior={}\n    if marker.exists():prior=json.loads(marker.read_text())\n    reuse=prior.get(\'source_identity\')==identity\n    totals={};copied=0;skipped=0\n    for split in ([\'train\',\'val\',\'test\'] if include_test else [\'train\',\'val\']):\n        src_images=source/split/\'images\';src_labels=source/split/\'labels\'\n        names=sorted(name for name in os.listdir(src_images) if name.lower().endswith(\'.jpg\'))\n        images=mapped/split/\'images\';labels=mapped/split/\'labels\'\n        images.mkdir(parents=True,exist_ok=True);labels.mkdir(parents=True,exist_ok=True)\n        progress.update(\'materialize\',split=split,completed=0,total=len(names),workers=workers,force=True)\n        def copy_one(name):\n            target=images/name;label=labels/(Path(name).stem+\'.txt\')\n            # is_symlink checks the local directory entry without following it\n            # back to FUSE; old remote symlinks are deliberately replaced.\n            if reuse and not target.is_symlink() and target.is_file() and label.is_file():return False\n            text=remap_labels((src_labels/label.name).read_text(encoding=\'utf-8\'))\n            temporary=target.with_name(target.name+\'.writing\')\n            try:\n                with (src_images/name).open(\'rb\') as incoming,temporary.open(\'wb\') as outgoing:\n                    shutil.copyfileobj(incoming,outgoing,length=1024*1024)\n                os.replace(temporary,target)\n                temporary_label=label.with_name(label.name+\'.writing\');temporary_label.write_text(text,encoding=\'utf-8\');os.replace(temporary_label,label)\n            finally:temporary.unlink(missing_ok=True)\n            return True\n        completed=0;iterator=iter(names)\n        with ThreadPoolExecutor(max_workers=workers,thread_name_prefix=\'fpi-local-copy\') as pool:\n            pending=set()\n            def fill():\n                while len(pending)<workers*2:\n                    try:name=next(iterator)\n                    except StopIteration:break\n                    pending.add(pool.submit(copy_one,name))\n            fill()\n            while pending:\n                done,pending=wait(pending,return_when=FIRST_COMPLETED)\n                for future in done:\n                    if future.result():copied+=1\n                    else:skipped+=1\n                    completed+=1\n                progress.update(\'materialize\',split=split,completed=completed,total=len(names),copied=copied,reused=skipped)\n                fill()\n        totals[split]=completed\n    result={\'source_identity\':identity,\'counts\':totals,\'copied\':copied,\'reused\':skipped,\n        \'images\':\'local regular files, byte-preserving copy\',\'test_prepared\':include_test}\n    atomic_json(marker,result);progress.update(\'materialize_complete\',counts=totals,force=True)\n    return result\n\n\ndef bounded_tail(path,limit=2048):\n    """Read at most limit bytes regardless of the tqdm log\'s total size."""\n    path=Path(path)\n    if not path.exists():return \'\'\n    with path.open(\'rb\') as stream:\n        stream.seek(0,2);stream.seek(max(0,stream.tell()-limit));return stream.read(limit).decode(\'utf-8\',errors=\'replace\')\n'}
for name,body in PROGRAM_FILES.items():
    (PROGRAM/name).write_text(body,encoding="utf-8")


In [ ]:
# Staging from FUSE is explicit, bounded and logged; no repeated remote per-frame reads.
SELECTED=['person','clothing'] if TASK=='both' else [TASK]
markers=list(Path('/kaggle/input').rglob('LINQ_TARGETED_INPUT.json'))
found={}
for marker in markers:
    meta=json.loads(marker.read_text())
    if meta['task'] in SELECTED:
        assert meta['task'] not in found,'Attach exactly one input version per task'
        found[meta['task']]=(marker.parent,meta)
for name in SELECTED:
    if name in found:continue
    archives=list(Path('/kaggle/input').rglob(f'linq-{name}-training-input.zip'))
    assert len(archives)==1,f'Attach exactly linq-{name}-training-input.zip as input'
    dest=Path('/kaggle/working/targeted-unpacked')/name;dest.mkdir(parents=True,exist_ok=True)
    print('UNPACK_START',name,archives[0].stat().st_size,flush=True)
    with zipfile.ZipFile(archives[0]) as z:
        for member in z.infolist():
            assert (dest/member.filename).resolve().is_relative_to(dest.resolve()),'Unsafe ZIP path'
        z.extractall(dest)
    meta=json.loads((dest/'LINQ_TARGETED_INPUT.json').read_text());assert meta['task']==name
    found[name]=(dest,meta)
    print('UNPACK_DONE',name,flush=True)
from concurrent.futures import ThreadPoolExecutor
STAGED={}
for name in SELECTED:
    source,meta=found[name];data=source/meta['data_directory']
    local=Path('/kaggle/temp/linq-targeted')/name;local.mkdir(parents=True,exist_ok=True)
    rows=[json.loads(line) for line in (data/'manifest.jsonl').read_text().splitlines() if line]
    selected=[r for r in rows if r['split'] in ('train','val')]
    jobs=[]
    for row in selected:
        src=data/row['file'];dst=local/row['file']
        assert dst.resolve().is_relative_to(local.resolve())
        jobs.append((src,dst,row.get('image_sha256')))
        if name=='person':
            rel=Path(row['file']);label=Path('labels')/rel.parent.name/rel.with_suffix('.txt').name
            jobs.append((data/label,local/label,None))
    def copy_one(job):
        src,dst,sha=job;dst.parent.mkdir(parents=True,exist_ok=True)
        with src.open('rb') as incoming,dst.open('wb') as outgoing:
            hasher=hashlib.sha256()
            while chunk:=incoming.read(1024*1024):outgoing.write(chunk);hasher.update(chunk)
        if sha:assert hasher.hexdigest()==sha,'Input image checksum mismatch'
    print('STAGING_START',name,len(jobs),flush=True)
    # map's result stream is consumed; errors cannot silently disappear.
    with ThreadPoolExecutor(max_workers=4) as pool:
        for i,_ in enumerate(pool.map(copy_one,jobs),1):
            if i%200==0:print('STAGING_PROGRESS',name,i,len(jobs),flush=True)
    (local/'manifest.jsonl').write_text(''.join(json.dumps(r)+'\n' for r in selected))
    weights=local/'initial.pt';shutil.copyfile(source/meta['weights_file'],weights)
    with weights.open('rb') as f:assert hashlib.file_digest(f,'sha256').hexdigest()==meta['initial_weights_sha256'],'Initial weights checksum mismatch'
    STAGED[name]={'data':local,'weights':weights}
    print('STAGING_VERIFIED',name,len(selected),'test_not_staged',sum(r['split']=='test' for r in rows),flush=True)


In [ ]:
from collections import deque
processes={};streams={};started={};statuses={}
for index,name in enumerate(SELECTED):
    output=RESULTS/name;output.mkdir(exist_ok=True)
    (output/'STOP').unlink(missing_ok=True)
    script='train_targeted_person.py' if name=='person' else 'train_signal_clothing.py'
    # The child's physical GPU is isolated; --device 0 is then its visible GPU.
    env={**os.environ,'CUDA_VISIBLE_DEVICES':str(index if TASK=='both' else 0)}
    command=[sys.executable,'-u',str(PROGRAM/script),'--data',str(STAGED[name]['data']),'--weights',str(STAGED[name]['weights']),
        '--output',str(output),'--minutes',str(TRAIN_MINUTES),'--device','0']
    if RESUME_CHECKPOINTS.get(name):command+=['--resume',str(RESUME_CHECKPOINTS[name])]
    streams[name]=(RESULTS/(name+'.log')).open('w')
    processes[name]=subprocess.Popen(command,stdout=streams[name],stderr=subprocess.STDOUT,env=env,cwd=PROGRAM)
    started[name]=time.monotonic();print('JOB_STARTED',name,processes[name].pid,'own_gpu',env['CUDA_VISIBLE_DEVICES'],flush=True)
def tail(path,size=1600):
    with path.open('rb') as f:f.seek(max(0,path.stat().st_size-size));return f.read().decode(errors='replace')
try:
    while len(statuses)<len(SELECTED):
        for name,process in processes.items():
            if name in statuses:continue
            age=(time.monotonic()-NOTEBOOK_STARTED)/60
            output=RESULTS/name;progress=output/'progress.json'
            if progress.exists():
                try:print('HEARTBEAT',name,progress.read_text(),flush=True)
                except (OSError,ValueError):print('PROGRESS_UPDATE_IN_FLIGHT',name,flush=True)
            else:print('PROCESS_ALIVE_WAITING_FOR_STAGE',name,round(time.monotonic()-started[name]),flush=True)
            print('BOUNDED_LOG_TAIL',name,tail(RESULTS/(name+'.log')),flush=True)
            if age>=270 and not (output/'STOP').exists():
                (output/'STOP').touch();print('GRACEFUL_STOP_REQUEST',name,'reserve validation/export',flush=True)
            if age>=285 and process.poll() is None:
                process.terminate();print('WALL_LIMIT_TERMINATE',name,'preserving existing checkpoints',flush=True)
                try:process.wait(timeout=20)
                except subprocess.TimeoutExpired:process.kill();process.wait()
            if process.poll() is not None:
                statuses[name]={'exit_code':process.returncode};streams[name].flush()
                print('JOB_FINISHED',name,process.returncode,flush=True)
        if len(statuses)<len(SELECTED):time.sleep(20)
finally:
    for name,process in processes.items():
        if process.poll() is None:
            process.terminate()
            try:process.wait(timeout=20)
            except subprocess.TimeoutExpired:process.kill();process.wait()
        streams[name].close()
    (RESULTS/'RUN_STATUS.json').write_text(json.dumps({'jobs':statuses,'models_installed':False,'remote_backup':False},indent=2))
    archive=Path('/kaggle/working/linq-targeted-trained-candidates.zip')
    with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=1) as z:
        for file in RESULTS.rglob('*'):
            if file.is_file() and not file.name.endswith('.writing'):z.write(file,file.relative_to(RESULTS))
    print('RESULT_ARCHIVE',str(archive),archive.stat().st_size,flush=True)
assert len(statuses)==len(SELECTED) and all(s['exit_code']==0 for s in statuses.values()),'At least one job failed; inspect its bounded log tail and preserve resume checkpoints from Outputs.'
